# 06 · Infrastructure for ML-built polyatomic Hamiltonians

Offline, CPU-only. Shows the pieces a learned proposer or amortized model plugs into:

1. **Operator grammar + polyad compiler** (`rd.grammar`, `rd.vib`): a term is text such as
   `k: sym(a1+ a4x a4x) + h.c.`; the compiler rejects anything non-Hermitian, non-symmetric,
   polyad-breaking or duplicated *before* it is fitted.
2. **Structure search** (`rd.selection`): candidate terms from an enumerator or any callable
   (e.g. a language model) are scored by Laplace evidence after a trial fit.
3. **Amortized inference** (`rd.ml`): simulated datasets, a calibrated fast posterior, and
   `verify` (a certified physics fit started from the amortized guess).
4. **Multi-fidelity ab initio** (`rd.abinitio`): Δ-GP from cheap + few expensive points, with
   the variance propagated to levels.

In [ ]:
import collections, os
from pathlib import Path
import numpy as np
import riss_drucker as rd
from riss_drucker import vib, selection
from riss_drucker.ml import dataset, amortized
from riss_drucker.abinitio import multifidelity, torsion
WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))

## 1 · Methane polyads from symmetry alone

In [ ]:
METHANE = [vib.Mode("1", "A1", 2), vib.Mode("2", "E", 1), vib.Mode("3", "F2", 2), vib.Mode("4", "F2", 1)]
base = vib.VibSpec(METHANE, "Td", [0, 1, 2, 3], ["w1: n1", "w2: n2", "w3: n3", "w4: n4"])
cands = selection.EnumerativeProposer(orders=(3, 4)).propose({"spec": base})
m = vib.compile_vib(vib.VibSpec(METHANE, "Td", [0, 1, 2, 3], base.terms + cands))
print(len(cands), "symmetry-allowed cubic/quartic terms; all compile certificates passed")
rng = np.random.default_rng(1)
th = dict(zip(m.param_names, [2917, 1534, 3019, 1306] + list(rng.normal(0, 5, len(m.param_names) - 4))))
sol = rd.solve(m, th, window=(-1e9, 1e9), method="dense")
for b, r in zip(sol.blocks, sol.results):
    e = np.round(np.sort(r["eigenvalues"]), 6)
    print(b["irrep"], "sublevels:", len(collections.Counter(e)))

## 2 · Structure search finds a Fermi resonance

In [ ]:
modes = [vib.Mode("1", "A1", 2), vib.Mode("2", "A1", 1)]
truth = vib.VibSpec(modes, "C1", [0, 1, 2, 3, 4], ["w1: n1", "w2: n2", "x22: n2*n2", "k: a1+ a2 a2 + h.c."])
tsol = rd.solve(vib.compile_vib(truth), {"w1": 3000.0, "w2": 1495.0, "x22": -4.0, "k": 30.0},
                window=(-1e9, 1e9), method="dense")
rng = np.random.default_rng(0)
data = [(b["irrep"], i, e + 0.01 * rng.standard_normal())
        for b, r in zip(tsol.blocks, tsol.results) for i, e in enumerate(r["eigenvalues"])]
def obs_for(model):
    keys = {b["irrep"]: b["key"] for b in model.blocks}
    return [rd.Observation(keys[p], i, e, 0.01) for p, i, e in data]
start = vib.VibSpec(modes, "C1", [0, 1, 2, 3, 4], ["w1: n1", "w2: n2", "x22: n2*n2"])
res = selection.search(start, obs_for, {"w1": 3000.0, "w2": 1495.0, "x22": -4.0},
                       ["k: a1+ a2 a2 + h.c.", "x11: n1*n1", "x12: n1*n2"], max_add=3)
for s in res.steps:
    print(f"{s.term:28s} accepted={s.accepted}")
print("k =", res.fit.theta["k"])

## 3 · Amortized posterior, then certified verification

In [ ]:
m = rd.compile_asym_rotor(j_max=4, order=2)
prior = dataset.Prior({"A": ("uniform", 2.5, 4.0), "B": ("uniform", 1.6, 2.4), "C": ("uniform", 0.8, 1.4)})
dataset.generate(m, prior, 300, WORK / "nb06_train", seed=1, n_levels=4, n_moments=8, noise=1e-3)
dataset.generate(m, prior, 100, WORK / "nb06_test", seed=2, n_levels=4, n_moments=8, noise=1e-3)
tr, te = dataset.load(WORK / "nb06_train"), dataset.load(WORK / "nb06_test")
post = amortized.BayesLinearPosterior().fit(tr["levels"], tr["theta"])
cal = amortized.calibration(post, te["levels"], te["theta"])
print("rmse per parameter:", np.round(cal["per_parameter_rmse"], 4), " coverage:", cal["empirical"])
truth = {"A": 3.3, "B": 2.1, "C": 1.05}
obs = rd.synth.synthesize_levels(m, truth, noise=1e-3, seed=4)
mean, _ = post.predict(dataset.level_features(m, truth, 4)[None, :])
v = amortized.verify(m, obs, dict(zip(["A", "B", "C"], mean[0])))
print("certified:", v["certified"], " theta:", {k: round(x, 5) for k, x in v["fit"].theta.items()})

## 4 · Δ-learning a torsional potential and propagating its variance

In [ ]:
f_high = lambda x: 400 * (1 - np.cos(x)) / 2 + 900 * (1 - np.cos(2 * x)) / 2 + 30 * np.sin(x) ** 2
f_low = lambda x: 380 * (1 - np.cos(x)) / 2 + 950 * (1 - np.cos(2 * x)) / 2
lx = np.linspace(0, 2 * np.pi, 72, endpoint=False); hx = np.linspace(0, 2 * np.pi, 9, endpoint=False)
dm = multifidelity.DeltaModel.fit(lx, f_low(lx), hx, f_high(hx), kind="periodic")
grid = np.linspace(0, 2 * np.pi, 180, endpoint=False)
model = rd.compile_torsion_rotor(n_fold=1, v_orders=[1, 2, 3, 4, 5, 6], m_max=40)
p = torsion.propagate(model, 20.0, grid, dm.sample(grid, 50, seed=1), [1, 2, 3, 4, 5, 6], nlowest=4)
exact = torsion.propagate(model, 20.0, grid, [f_high(grid)], [1, 2, 3, 4, 5, 6], nlowest=4)["mean"]
for i in range(1, 5):
    print(f"level {i}: {p['mean'][i]:9.3f} ± {p['sd'][i]:.3f}   (exact {exact[i]:9.3f})")